In [16]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from ocr_vs_vlm.results_final.shared.colors import MODEL_COLORS, get_model_color, get_dataset_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import bootstrap_ci, wilcoxon_test, cohens_d
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template, create_metric_boxplot
from ocr_vs_vlm.results_final.shared.data_loader import load_experiment_data, extract_models_from_columns

setup_plotly_template()
print("✓ Setup complete")

✓ Setup complete


## 1. Load Parsing Experiment Data

In [17]:
PARSING_PHASES = ['phase_1', 'phase_2', 'phase_3']
DATASETS = ['IAM_mini', 'ICDAR_mini', 'VOC2007']

all_data = []
for dataset in DATASETS:
    for phase in PARSING_PHASES:
        try:
            df = load_experiment_data(dataset, phase)
            all_data.append(df)
            print(f"✓ {dataset}/{phase}: {len(df)} samples")
        except FileNotFoundError:
            # Try phase_3a variant
            try:
                df = load_experiment_data(dataset, phase.replace('phase_3', 'phase_3a'))
                df['phase'] = phase  # Normalize
                all_data.append(df)
                print(f"✓ {dataset}/{phase} (as 3a): {len(df)} samples")
            except FileNotFoundError:
                print(f"✗ Missing {dataset}/{phase}")

df_parsing = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
print(f"\n📊 Total: {len(df_parsing)} samples")

✓ IAM_mini/phase_1: 500 samples
✓ IAM_mini/phase_2: 500 samples
✓ IAM_mini/phase_3: 500 samples
✓ ICDAR_mini/phase_1: 491 samples
✓ ICDAR_mini/phase_2: 491 samples
✓ ICDAR_mini/phase_3 (as 3a): 491 samples
✓ VOC2007/phase_1: 238 samples
✓ VOC2007/phase_2: 238 samples
✓ VOC2007/phase_3 (as 3a): 238 samples

📊 Total: 3687 samples


In [18]:
df.head()

,sample_id,image_path,ground_truth,language,dataset,prediction_claude_sonnet,inference_time_ms_claude_sonnet,tokens_used_claude_sonnet,error_claude_sonnet,prediction_gpt-5-mini,inference_time_ms_gpt-5-mini,tokens_used_gpt-5-mini,error_gpt-5-mini,prediction_gpt-5-nano,inference_time_ms_gpt-5-nano,tokens_used_gpt-5-nano,error_gpt-5-nano,phase,approach
0,voc2007_illu_item10+_D_1,/Users/kenzabenkirane/Documents/GitHub/researc...,报告时间：\n报告类型：肾功能检查\n报告者签名：\n审核者签名：\n说明：该报告的数据仅对...,zh-CN,VOC2007,NaN,3581.202269,NaN,NaN,报告时间： 报告类型： 肾功能检查\nID号： ...,35408.864975,NaN,NaN,NaN,27068.521976,NaN,NaN,phase_3,vlm_task_aware
1,voc2007_illu_item10+_D_2,/Users/kenzabenkirane/Documents/GitHub/researc...,报告时间：\n报告类型：血气试验（含钾钠氯）\n报告者签名：\n审核者签名：\n说明：该报告...,zh-CN,VOC2007,NaN,3613.608122,NaN,NaN,报告时间：\n报告类型：血气试验(含抑制钠氯)\n姓 名：\n性 别：\n年 龄：\n序号/...,20668.776035,NaN,NaN,NaN,26910.904169,NaN,NaN,phase_3,vlm_task_aware
2,voc2007_illu_item10+_D_3,/Users/kenzabenkirane/Documents/GitHub/researc...,报告时间：\n报告类型：血气试验（含钾钠氯）\n报告者签名：\n审核者签名：\n说明：该报告...,zh-CN,VOC2007,NaN,3599.278927,NaN,NaN,报告时间： 报告类型：血气试验（含电解物）...,35462.339878,NaN,NaN,NaN,27944.575071,NaN,NaN,phase_3,vlm_task_aware
3,voc2007_illu_item10+_D_4,/Users/kenzabenkirane/Documents/GitHub/researc...,报告时间：\n报告类型：血气试验（含钾钠氯）\n报告者签名：\n审核者签名：\n说明：该报告...,zh-CN,VOC2007,NaN,3587.430000,NaN,NaN,报告时间： 报告类型：血气试验（含电解...,28606.616974,NaN,NaN,NaN,25769.673109,NaN,NaN,phase_3,vlm_task_aware
4,voc2007_illu_item10+_D_5,/Users/kenzabenkirane/Documents/GitHub/researc...,报告时间：\n报告类型：尿液分析（尿干化学）\n报告者签名：\n审核者签名：\n说明：该报告...,zh-CN,VOC2007,NaN,3583.699942,NaN,NaN,报告时间： 报告类型： 尿液分析（尿干化学）\n\...,36387.460709,NaN,NaN,NaN,28216.312170,NaN,NaN,phase_3,vlm_task_aware


## 2. OCR vs VLM Comparison (CER)

In [19]:
# Extract CER scores
models = extract_models_from_columns(df_parsing)

cer_by_approach = {'P-OCR': [], 'P-VLM_simple': [], 'P-VLM_taskaware': []}
approach_map = {'phase_1': 'P-OCR', 'phase_2': 'P-VLM_simple', 'phase_3': 'P-VLM_taskaware'}

stats_rows = []
for phase in PARSING_PHASES:
    phase_df = df_parsing[df_parsing['phase'] == phase]
    approach = approach_map[phase]
    
    for model in models:
        col = f'cer_{model}'
        if col not in phase_df.columns:
            col = f'prediction_{model}'  # Fallback - would need metric calculation
            continue
        
        scores = phase_df[col].dropna().values
        if len(scores) > 0:
            mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
            stats_rows.append({
                'Approach': approach,
                'Model': model,
                'CER': mean,
                'CI_Lower': ci_lo,
                'CI_Upper': ci_hi
            })

stats_df = pd.DataFrame(stats_rows)
if len(stats_df) > 0:
    display(stats_df.pivot_table(index='Model', columns='Approach', values='CER').round(4))

In [20]:
# Visualization: CER by approach
if len(stats_df) > 0:
    fig = go.Figure()
    
    for approach in stats_df['Approach'].unique():
        app_data = stats_df[stats_df['Approach'] == approach]
        color = APPROACH_COLORS.get('ocr_baseline' if 'OCR' in approach else 'vlm_generic', '#6B7280')
        
        fig.add_trace(go.Bar(
            x=app_data['Model'],
            y=app_data['CER'],
            name=approach,
            marker_color=color,
            error_y=dict(type='data', array=app_data['CER'] - app_data['CI_Lower'])
        ))
    
    fig.update_layout(
        title='Parsing: CER by Approach (Lower is Better)',
        yaxis_title='Character Error Rate (CER)',
        barmode='group',
        height=500
    )
    fig.show()
else:
    print("⚠️ No CER data available - check column naming in results files")

⚠️ No CER data available - check column naming in results files


## 3. Dataset Comparison

In [21]:
# Compare across datasets
dataset_stats = []
for dataset in DATASETS:
    ds_df = df_parsing[df_parsing['dataset'] == dataset]
    for model in models:
        for metric in ['cer', 'wer']:
            col = f'{metric}_{model}'
            if col in ds_df.columns:
                scores = ds_df[col].dropna().values
                if len(scores) > 0:
                    mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                    dataset_stats.append({
                        'Dataset': dataset,
                        'Model': model,
                        'Metric': metric.upper(),
                        'Value': mean
                    })

ds_df = pd.DataFrame(dataset_stats)
if len(ds_df) > 0:
    pivot = ds_df.pivot_table(index=['Dataset', 'Model'], columns='Metric', values='Value')
    display(pivot.round(4))

## 4. Summary

In [22]:
print("=" * 70)
print("PARSING EXPERIMENTS SUMMARY")
print("=" * 70)

if len(stats_df) > 0:
    # Best by approach
    print("\n📊 Best CER by Approach:")
    for approach in stats_df['Approach'].unique():
        best = stats_df[stats_df['Approach'] == approach].nsmallest(1, 'CER').iloc[0]
        print(f"   {approach}: {best['Model']} → CER={best['CER']:.4f}")
    
    # Overall best
    overall_best = stats_df.nsmallest(1, 'CER').iloc[0]
    print(f"\n🏆 Overall Best: {overall_best['Model']} ({overall_best['Approach']}) → CER={overall_best['CER']:.4f}")

print("\n" + "=" * 70)

PARSING EXPERIMENTS SUMMARY

